# RAG vs No-RAG Comparison (Features + Test Cases)

This notebook runs a side-by-side comparison on one SRS document.

Important fairness notes:
- Feature extraction in the current codebase is **not retrieval-based**; therefore, features are extracted once and reused in both modes.
- The core comparison here is on **test-case generation per feature**.
- For reproducibility, generation uses `temperature=0.0`.
- To keep baseline fair, No-RAG can be run with **matched context budget** (`random_k_matched`) so both modes receive comparable context size.

Modes:
- **RAG**: per-feature top-k semantic retrieval context.
- **No-RAG**:
  - `feature_only`: only feature text
  - `global_context`: fixed global document context
  - `random_k_matched`: random top-k chunks with the same context budget as RAG (recommended baseline)

Default target:
`data/srs/JDECo_SRS.docx[1].pdf`

In [10]:
from __future__ import annotations

import json
import random
import sys
from pathlib import Path

import pandas as pd

In [11]:
# Resolve project paths
NOTEBOOK = Path.cwd()
if NOTEBOOK.name != "notebooks":
    # If launched from repo root, adjust to notebooks path
    candidate = NOTEBOOK / "QBrain" / "rag_lab" / "notebooks"
    if candidate.exists():
        NOTEBOOK = candidate

RAG_LAB = NOTEBOOK.parent
SRC = RAG_LAB / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qbrain_rag.application.chunking import chunk_text
from qbrain_rag.application.document_context import documents_to_context_string
from qbrain_rag.application.feature_extraction import extract_features_from_indexed_chunks
from qbrain_rag.application.prompts_srs import build_test_case_user_prompt
from qbrain_rag.config.settings import get_settings
from qbrain_rag.infrastructure.document_loaders import load_document
from qbrain_rag.infrastructure.llm import complete_json_object
from qbrain_rag.infrastructure.vector_store import build_faiss_store, list_all_documents_ordered

print("RAG_LAB:", RAG_LAB)
print("SRC:", SRC)

RAG_LAB: d:\Qbrainpython\QBrain\rag_lab
SRC: d:\Qbrainpython\QBrain\rag_lab\src


In [12]:
# Configuration (minimal)
s = get_settings()
DOC_PATH = RAG_LAB / "data" / "srs" / "JDECo_SRS.docx[1].pdf"
MAX_FEATURES = None                    # None = use all extracted features
TOP_K = s.default_top_k                # shared for RAG and matched baseline
CONTEXT_BUDGET = 24000                 # shared context limit for both modes
NO_RAG_MODE = "random_k_matched"      # "feature_only" | "global_context" | "random_k_matched"
TEMPERATURE = 0.0
OUT_DIR = RAG_LAB / "results" / "tables" / "rag_vs_no_rag"

# Only used in random baseline mode
RANDOM_SEED = 42

OUT_DIR.mkdir(parents=True, exist_ok=True)
assert DOC_PATH.exists(), f"Missing file: {DOC_PATH}"
assert TOP_K >= 1, "TOP_K must be >= 1"

print("Document:", DOC_PATH)
print("Output dir:", OUT_DIR)
print("MAX_FEATURES:", MAX_FEATURES)
print("TOP_K:", TOP_K)
print("CONTEXT_BUDGET:", CONTEXT_BUDGET)
print("NO_RAG_MODE:", NO_RAG_MODE)
print("Temperature:", TEMPERATURE)

Document: d:\Qbrainpython\QBrain\rag_lab\data\srs\JDECo_SRS.docx[1].pdf
Output dir: d:\Qbrainpython\QBrain\rag_lab\results\tables\rag_vs_no_rag
MAX_FEATURES: None
TOP_K: 5
CONTEXT_BUDGET: 24000
NO_RAG_MODE: random_k_matched
Temperature: 0.0


In [13]:
# Build index and extract features from full document context
text = load_document(str(DOC_PATH))
chunks = chunk_text(text)
metadata = [{"source_file": DOC_PATH.name, "chunk_id": i + 1} for i in range(len(chunks))]
store = build_faiss_store(chunks, metadata)

bundle = extract_features_from_indexed_chunks(store)
all_features = bundle.get("features", []) if isinstance(bundle, dict) else []
if not isinstance(all_features, list):
    all_features = []

features = all_features if MAX_FEATURES is None else all_features[:MAX_FEATURES]
print(f"Chunks: {len(chunks)}")
print(f"Extracted features (total): {len(all_features)}")
print(f"Using features: {len(features)}")
print("Feature extraction note: features are shared across both modes in this notebook.")

Chunks: 30
Extracted features (total): 9
Using features: 9
Feature extraction note: features are shared across both modes in this notebook.


In [14]:
def _normalize_testcases(raw_list):
    normalized = []
    if not isinstance(raw_list, list):
        return normalized
    for i, tc in enumerate(raw_list):
        if not isinstance(tc, dict):
            continue
        title = str(tc.get("title") or "").strip()
        if not title:
            continue
        normalized.append(
            {
                "testCaseId": str(tc.get("testCaseId") or f"TC_{i + 1:03d}"),
                "title": title,
                "description": str(tc.get("description") or ""),
                "steps": [str(s) for s in tc.get("steps", [])] if isinstance(tc.get("steps"), list) else [],
                "expectedResult": str(tc.get("expectedResult") or ""),
                "priority": str(tc.get("priority") or "medium").lower(),
                "status": str(tc.get("status") or "pending"),
                "preconditions": [str(p) for p in tc.get("preconditions", [])]
                if isinstance(tc.get("preconditions"), list)
                else [],
                "testData": tc.get("testData") if isinstance(tc.get("testData"), dict) else {},
            }
        )
    return normalized


def _feature_query(feature: dict) -> str:
    name = str(feature.get("name") or "").strip()
    desc = str(feature.get("description") or "").strip()
    matched = feature.get("matchedSections") or []
    if not isinstance(matched, list):
        matched = []
    feature_description = f"{name}\n{desc}".strip() or name or desc
    parts = [feature_description[:3000]]
    if matched:
        parts.insert(0, " ".join(str(s) for s in matched[:12]))
    return " ".join(parts).strip()


_TC_SYSTEM = (
    "You are an expert QA engineer. Generate concrete, testable test cases grounded in the user's SOURCE CONTEXT only. "
    "The text may be unstructured; do not assume sections or IDs unless they appear in the context. "
    "Obey the JSON shape in the user message exactly. Output one JSON object only, no markdown."
)

# Guard against out-of-order notebook execution
try:
    list_all_documents_ordered
except NameError:
    try:
        from qbrain_rag.infrastructure.vector_store import list_all_documents_ordered
    except ModuleNotFoundError:
        # Fallback: resolve rag_lab/src and add to sys.path
        from pathlib import Path
        import sys

        p = Path.cwd()
        if p.name == "notebooks":
            rag_lab = p.parent
        elif (p / "QBrain" / "rag_lab" / "src").exists():
            rag_lab = p / "QBrain" / "rag_lab"
        else:
            rag_lab = p
        src = rag_lab / "src"
        if str(src) not in sys.path:
            sys.path.insert(0, str(src))
        from qbrain_rag.infrastructure.vector_store import list_all_documents_ordered

if "store" not in globals():
    # Auto-build store if indexing cell was skipped
    try:
        load_document
        chunk_text
        build_faiss_store
    except NameError:
        from qbrain_rag.application.chunking import chunk_text
        from qbrain_rag.infrastructure.document_loaders import load_document
        from qbrain_rag.infrastructure.vector_store import build_faiss_store

    if "DOC_PATH" not in globals():
        p = Path.cwd()
        if p.name == "notebooks":
            rag_lab = p.parent
        elif (p / "QBrain" / "rag_lab" / "data" / "srs").exists():
            rag_lab = p / "QBrain" / "rag_lab"
        else:
            rag_lab = p
        DOC_PATH = rag_lab / "data" / "srs" / "JDECo_SRS.docx[1].pdf"
        print(f"DOC_PATH was missing. Using default: {DOC_PATH}")

    text = load_document(str(DOC_PATH))
    chunks = chunk_text(text)
    metadata = [{"source_file": Path(DOC_PATH).name, "chunk_id": i + 1} for i in range(len(chunks))]
    store = build_faiss_store(chunks, metadata)
    print(f"Auto-built store from DOC_PATH with {len(chunks)} chunks.")

# Ensure local deps/constants exist if cell is executed standalone
try:
    random
except NameError:
    import random

if "RANDOM_SEED" not in globals():
    RANDOM_SEED = 42
if "CONTEXT_BUDGET" not in globals():
    CONTEXT_BUDGET = 24000

all_docs = list_all_documents_ordered(store)
rng = random.Random(RANDOM_SEED)

# Shared no-rag context (used only in global_context mode)
NO_RAG_CONTEXT, _, _ = documents_to_context_string(all_docs, max_chars=CONTEXT_BUDGET)
print("No-RAG global context chars:", len(NO_RAG_CONTEXT))

No-RAG global context chars: 22751


In [15]:
results = []

# Guard against out-of-order execution
if "features" not in globals():
    if "all_features" in globals():
        features = all_features if MAX_FEATURES is None else all_features[:MAX_FEATURES]
    elif "store" in globals():
        bundle = extract_features_from_indexed_chunks(store)
        all_features = bundle.get("features", []) if isinstance(bundle, dict) else []
        if not isinstance(all_features, list):
            all_features = []
        features = all_features if MAX_FEATURES is None else all_features[:MAX_FEATURES]
    else:
        raise RuntimeError("Missing prerequisites: run setup cells (imports/config/indexing) first.")

for idx, feature in enumerate(features, start=1):
    name = str(feature.get("name") or f"feature_{idx}")
    desc = str(feature.get("description") or "")
    ftype = str(feature.get("featureType") or "FUNCTIONAL")
    matched = feature.get("matchedSections") or []
    if not isinstance(matched, list):
        matched = []

    feature_description = f"{name}\n{desc}".strip() or name

    # RAG mode context (per-feature semantic retrieval)
    q = _feature_query(feature)
    rag_docs = store.similarity_search(q, k=TOP_K)
    rag_context, _, _ = documents_to_context_string(rag_docs, max_chars=CONTEXT_BUDGET)
    rag_prompt = build_test_case_user_prompt(
        feature_description=feature_description,
        context=rag_context,
        feature_type=ftype,
        matched_sections=[str(x) for x in matched],
    )
    rag_parsed = complete_json_object(_TC_SYSTEM, rag_prompt, temperature=TEMPERATURE)
    rag_testcases = _normalize_testcases(rag_parsed.get("testCases"))

    # No-RAG baseline context
    if NO_RAG_MODE == "feature_only":
        no_rag_context = feature_description
    elif NO_RAG_MODE == "global_context":
        no_rag_context = NO_RAG_CONTEXT
    elif NO_RAG_MODE == "random_k_matched":
        sample_k = min(TOP_K, len(all_docs))
        sampled_docs = rng.sample(all_docs, k=sample_k) if sample_k > 0 else []
        no_rag_context, _, _ = documents_to_context_string(sampled_docs, max_chars=CONTEXT_BUDGET)
    else:
        raise ValueError(f"Unsupported NO_RAG_MODE: {NO_RAG_MODE}")

    no_rag_prompt = build_test_case_user_prompt(
        feature_description=feature_description,
        context=no_rag_context,
        feature_type=ftype,
        matched_sections=[str(x) for x in matched],
    )
    no_rag_parsed = complete_json_object(_TC_SYSTEM, no_rag_prompt, temperature=TEMPERATURE)
    no_rag_testcases = _normalize_testcases(no_rag_parsed.get("testCases"))

    results.append(
        {
            "feature_index": idx,
            "feature_name": name,
            "feature_type": ftype,
            "matched_sections_count": len(matched),
            "rag_testcases": rag_testcases,
            "no_rag_testcases": no_rag_testcases,
            "rag_count": len(rag_testcases),
            "no_rag_count": len(no_rag_testcases),
            "delta_count": len(rag_testcases) - len(no_rag_testcases),
        }
    )

print(f"Done. Compared {len(results)} features.")

run_meta = {
    "document": str(DOC_PATH),
    "max_features": MAX_FEATURES,
    "top_k": TOP_K,
    "context_budget": CONTEXT_BUDGET,
    "no_rag_mode": NO_RAG_MODE,
    "random_seed": RANDOM_SEED,
    "temperature": TEMPERATURE,
    "features_compared": len(results),
}

with open(OUT_DIR / "rag_vs_no_rag_raw.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

with open(OUT_DIR / "rag_vs_no_rag_run_meta.json", "w", encoding="utf-8") as f:
    json.dump(run_meta, f, ensure_ascii=False, indent=2)

print("Saved:", OUT_DIR / "rag_vs_no_rag_raw.json")
print("Saved:", OUT_DIR / "rag_vs_no_rag_run_meta.json")

Done. Compared 9 features.
Saved: d:\Qbrainpython\QBrain\rag_lab\results\tables\rag_vs_no_rag\rag_vs_no_rag_raw.json
Saved: d:\Qbrainpython\QBrain\rag_lab\results\tables\rag_vs_no_rag\rag_vs_no_rag_run_meta.json


In [19]:
def _schema_valid(tc: dict) -> bool:
    required = [
        "testCaseId",
        "title",
        "description",
        "steps",
        "expectedResult",
        "priority",
        "status",
        "preconditions",
        "testData",
    ]
    if not isinstance(tc, dict):
        return False
    return all(k in tc for k in required)


def _completeness(tc: dict) -> float:
    score = 0
    score += 1 if isinstance(tc.get("steps"), list) and len(tc.get("steps", [])) >= 3 else 0
    score += 1 if str(tc.get("expectedResult") or "").strip() else 0
    score += 1 if isinstance(tc.get("preconditions"), list) and len(tc.get("preconditions", [])) > 0 else 0
    return score / 3.0


def _mode_metrics(mode_key: str) -> dict:
    all_tcs = []
    for row in results:
        all_tcs.extend(row[mode_key])
    n = len(all_tcs)
    if n == 0:
        return {
            "testcases_total": 0,
            "schema_valid_rate": 0.0,
            "avg_completeness": 0.0,
            "avg_steps": 0.0,
            "steps_ge_3_rate": 0.0,
        }

    schema_rate = sum(_schema_valid(tc) for tc in all_tcs) / n
    avg_comp = sum(_completeness(tc) for tc in all_tcs) / n
    avg_steps = sum(len(tc.get("steps", [])) for tc in all_tcs) / n
    steps_ge_3_rate = sum(len(tc.get("steps", [])) >= 3 for tc in all_tcs) / n

    return {
        "testcases_total": n,
        "schema_valid_rate": schema_rate,
        "avg_completeness": avg_comp,
        "avg_steps": avg_steps,
        "steps_ge_3_rate": steps_ge_3_rate,
    }


def _feature_level_stats() -> dict:
    if not results:
        return {"wins": 0, "ties": 0, "losses": 0, "mean_delta_count": 0.0, "median_delta_count": 0.0}

    deltas = [r["delta_count"] for r in results]
    wins = sum(d > 0 for d in deltas)
    ties = sum(d == 0 for d in deltas)
    losses = sum(d < 0 for d in deltas)

    s = pd.Series(deltas)
    return {
        "wins": wins,
        "ties": ties,
        "losses": losses,
        "mean_delta_count": float(s.mean()),
        "median_delta_count": float(s.median()),
    }

rag_metrics = _mode_metrics("rag_testcases")
no_rag_metrics = _mode_metrics("no_rag_testcases")
feature_stats = _feature_level_stats()

summary = pd.DataFrame(
    [
        {"metric": "features_compared", "rag": len(results), "no_rag": len(results)},
        {"metric": "testcases_total", "rag": rag_metrics["testcases_total"], "no_rag": no_rag_metrics["testcases_total"]},
        {"metric": "schema_valid_rate", "rag": rag_metrics["schema_valid_rate"], "no_rag": no_rag_metrics["schema_valid_rate"]},
        {"metric": "avg_completeness", "rag": rag_metrics["avg_completeness"], "no_rag": no_rag_metrics["avg_completeness"]},
        {"metric": "avg_steps", "rag": rag_metrics["avg_steps"], "no_rag": no_rag_metrics["avg_steps"]},
        {"metric": "steps_ge_3_rate", "rag": rag_metrics["steps_ge_3_rate"], "no_rag": no_rag_metrics["steps_ge_3_rate"]},
        {"metric": "feature_win_count_by_tc_count", "rag": feature_stats["wins"], "no_rag": feature_stats["losses"]},
        {"metric": "feature_tie_count_by_tc_count", "rag": feature_stats["ties"], "no_rag": feature_stats["ties"]},
        {"metric": "feature_mean_delta_tc_count", "rag": feature_stats["mean_delta_count"], "no_rag": -feature_stats["mean_delta_count"]},
        {"metric": "feature_median_delta_tc_count", "rag": feature_stats["median_delta_count"], "no_rag": -feature_stats["median_delta_count"]},
    ]
)
summary["delta_rag_minus_no_rag"] = summary["rag"] - summary["no_rag"]

per_feature = pd.DataFrame(
    [
        {
            "feature_index": r["feature_index"],
            "feature_name": r["feature_name"],
            "feature_type": r["feature_type"],
            "matched_sections_count": r["matched_sections_count"],
            "rag_count": r["rag_count"],
            "no_rag_count": r["no_rag_count"],
            "delta_count": r["delta_count"],
            "winner_by_count": "rag" if r["delta_count"] > 0 else ("no_rag" if r["delta_count"] < 0 else "tie"),
        }
        for r in results
    ]
)

summary.to_csv(OUT_DIR / "rag_vs_no_rag_summary.csv", index=False)
per_feature.to_csv(OUT_DIR / "rag_vs_no_rag_per_feature.csv", index=False)

print("Saved:", OUT_DIR / "rag_vs_no_rag_summary.csv")
print("Saved:", OUT_DIR / "rag_vs_no_rag_per_feature.csv")
print("Note: features are shared between both modes in current architecture.")
summary

Saved: d:\Qbrainpython\QBrain\rag_lab\results\tables\rag_vs_no_rag\rag_vs_no_rag_summary.csv
Saved: d:\Qbrainpython\QBrain\rag_lab\results\tables\rag_vs_no_rag\rag_vs_no_rag_per_feature.csv
Note: features are shared between both modes in current architecture.


,metric,rag,no_rag,delta_rag_minus_no_rag
0,features_compared,9.000000,9.000000,0.000000
1,testcases_total,50.000000,43.000000,7.000000
2,schema_valid_rate,1.000000,1.000000,0.000000
3,avg_completeness,0.913333,0.906977,0.006357
4,avg_steps,4.240000,4.186047,0.053953
5,steps_ge_3_rate,1.000000,0.976744,0.023256
6,feature_win_count_by_tc_count,6.000000,0.000000,6.000000
7,feature_tie_count_by_tc_count,3.000000,3.000000,0.000000
8,feature_mean_delta_tc_count,0.777778,-0.777778,1.555556
9,feature_median_delta_tc_count,1.000000,-1.000000,2.000000


In [17]:
per_feature.head(20)

,feature_index,feature_name,feature_type,matched_sections_count,rag_count,no_rag_count,delta_count,winner_by_count
0,1,Service Request Management,FUNCTIONAL,2,5,5,0,tie
1,2,Service Status Notifications,NOTIFICATION,2,6,5,1,rag
2,3,User Account Management,FUNCTIONAL,1,6,5,1,rag
3,4,Inventory Management,DATA_MODEL,1,5,4,1,rag
4,5,Payment Processing,FUNCTIONAL,2,5,5,0,tie
5,6,Service Installation Workflow,WORKFLOW,2,5,5,0,tie
6,7,User Interface Requirements,INTERFACE,1,7,6,1,rag
7,8,Data Retention Policies,DATA,1,5,3,2,rag
8,9,Conflict Resolution Mechanism,WORKFLOW,1,6,5,1,rag


## How This Notebook Works

1. **Load SRS** from `DOC_PATH` and split into chunks.
2. **Build vector index** (FAISS) from those chunks.
3. **Extract features once** from full document context (shared input for fairness).
4. For each feature, generate test cases in two modes:
   - **RAG:** retrieve top-k relevant chunks and generate from that context.
   - **No-RAG:** baseline context from one of:
     - `feature_only`
     - `global_context`
     - `random_k_matched` (recommended for fair comparison: same k and context budget as RAG)
5. Normalize outputs into a consistent test-case schema.
6. Compute:
   - aggregate metrics,
   - feature-level paired deltas,
   - win/tie/loss counts.
7. Save CSV/JSON outputs.

### Output files
- `rag_vs_no_rag_raw.json`: raw per-feature outputs for both modes.
- `rag_vs_no_rag_run_meta.json`: run configuration for reproducibility.
- `rag_vs_no_rag_summary.csv`: aggregate metrics + paired stats.
- `rag_vs_no_rag_per_feature.csv`: per-feature testcase count comparison + winner label.

In [18]:
# Optional: inspect one feature side-by-side
if results:
    i = 0
    sample = results[i]
    print("Feature:", sample["feature_name"])
    print("RAG testcases:", sample["rag_count"])
    print("No-RAG testcases:", sample["no_rag_count"])
    print("\nRAG sample title:", (sample["rag_testcases"][0]["title"] if sample["rag_testcases"] else "<none>"))
    print("No-RAG sample title:", (sample["no_rag_testcases"][0]["title"] if sample["no_rag_testcases"] else "<none>"))
else:
    print("No features/results available. Check extraction step and API key setup.")

Feature: Service Request Management
RAG testcases: 5
No-RAG testcases: 5

RAG sample title: Verify successful request for new 1-phase subscription
No-RAG sample title: Verify successful service request submission (Request a service from JDECo)
